# Grid-cell satellite features

Build the satellite features (tropospheric NO2 and the absorbing aerosol index, monthly means) for every cell of the analysis grid (Phase K). Notebook 03 only produced them at the station points. The definitive model from notebook 09, a Ridge regression, needs the same features in every cell to predict PM2.5 where there is no station (Phase L). The grid has about 435,000 cells, far too many for the interactive `reduceRegions` call used for the stations, so the monthly composites are exported as rasters and aggregated onto the grid locally.

In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
from pathlib import Path

import dotenv
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import xarray as xr
import rioxarray
import yaml

import ee
import geemap

import monitoring_deserts.data.fetch_earthengine as fetch_ee

## Configuration

Read the configuration parameters from the previous steps and the general project config. Initialize the Earth Engine client.

In [ ]:
# Generic settings
with open("../configs/project_config.yaml") as f:
    gen_config = yaml.safe_load(f)

RAW_DATA_DIR = Path(gen_config["data_dir"]["raw"]) / "satellite" / "grid" # exported rasters, one per month and band
CLEAN_DATA_DIR = Path(gen_config["data_dir"]["clean"]) / "satellite"
PROCESSED_DATA_DIR = Path(gen_config["data_dir"]["processed"]) # The data/processed/ should have no subdirs
MAPS_DIR = Path(gen_config["output_dir"]["maps"]) / "grid_satellite"
PLOTS_DIR = Path(gen_config["output_dir"]["plots"]) / "grid_satellite"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Spacetime settings
with open("../configs/census_scope.yaml") as f:
    census_config = yaml.safe_load(f)

BOUNDING_BOX = tuple(census_config["bounding_box"].values())
DAY_START, DAY_END = census_config["period"]["start"], census_config["period"]["end"]

# Satellite settings of notebook 03: same collections, bands and scale, so station and grid features match
with open("../configs/satellite_metadata.yaml") as f:
    sat_config = yaml.safe_load(f)

REDUCER_SCALE = sat_config["reducer_scale_m"]
IMAGE_COLLECTIONS = {name: spec.split(":") for name, spec in sat_config["image_collections"].items()} # name -> [collection id, band]

# The analysis grid from notebook 04 (cell_id, centroid_lon, centroid_lat, geometry)
analysis_grid = gpd.read_parquet(PROCESSED_DATA_DIR / "grid.parquet")

# Monthly windows, built exactly as in notebook 03
m_win = fetch_ee.build_month_windows(DAY_START, DAY_END)

# Initialize client
dotenv.load_dotenv()
ee.Initialize(project="monitoring-deserts")

## Outline

1. **Step 1 - Build the monthly composite images.** Reuse the collections, bands and monthly windows of notebook 03 (`fetch_ee.build_month_windows`) and take the mean of each band over each month, so a station and a grid cell are measured the same way. That gives one image per month and band, 32 in all (16 months, NO2 and AAI).
2. **Step 2 - Choose the export route and test it on one month.** The plan exports each composite with `Export.image.toDrive`: an asynchronous batch job that needs `maxPixels` raised and a download from Drive afterwards. The repository's `fetch_ee.download_image_on_grid` instead downloads small tiles already averaged onto the grid, with no Drive step, but it needs an image with a fixed projection and a monthly mean composite has none. Try one month with one route and check the shape of the result, the share of missing cells and the range of the values against the station values from notebook 03.
3. **Step 3 - Export all months and bands.** Run the chosen route for the 16 months and 2 bands. Cache each result in `RAW_DATA_DIR` as soon as it is done, as notebook 03 does with its temporary files, so an interrupted run can be resumed, and poll the export tasks if Drive exports are used.
4. **Step 4 - Aggregate onto the grid.** For each month, take the mean and the spread of the roughly 1 km pixels inside every 0.05 degree cell. The plan uses `rasterstats.zonal_stats`, which is not installed in the `mon-des` environment; the average resampling of `rioxarray` is the alternative. Name the columns as in the station table (`cell_id`, `month`, `no2`, `no2_sd`, `aai`, `aai_sd`) so the model's feature names line up.
5. **Step 5 - Check the result.** Look at the share of missing values per month (July 2025 was the worst month for NO2 at the stations), compare the distribution of the grid features with that of the stations (the model will be applied in places that may differ from where the stations sit), and compare each station's value with the value of the cell that contains it.
6. **Step 6 - Assemble the grid feature table for Phase L.** Join the satellite features with the gridded ERA5 meteorology from notebook 05 and the population layers from notebook 06, add `sin_month` and `cos_month`, and check that the columns are exactly the features of the definitive model recorded under `definitive_model` in `configs/model_training_metadata.yaml`.
7. **Step 7 - Save.** Write the monthly grid satellite features to `data/clean/satellite/` as parquet and the metadata to `configs/grid_satellite_metadata.yaml` (export route, scale, months, share of missing cells, file paths). Phase L (notebook 11) loads the fitted Ridge from `data/models/ridge_best.joblib` and predicts on the assembled table.